
# NB_04 — Claims Incremental Bronze-to-Silver Processing

## Purpose

This notebook implements production-style incremental processing for insurance Claims data from the Bronze Lakehouse into the Silver Lakehouse.

The processing pattern reuses the metadata-driven ETL framework established for Customers and Policies.

## Processing Flow

Bronze Claims  
→ Read ETL Control Metadata  
→ Read Stored Watermark  
→ Incremental Extraction  
→ Data Quality Validation  
→ Reject Invalid Records  
→ Deduplicate by Claim Business Key  
→ Standardize Silver Schema  
→ Detect INSERT / UPDATE / NO-OP  
→ Delta MERGE into Silver  
→ Write Batch Audit  
→ Advance Watermark  
→ Verify Restart / Idempotency

## Business Key

`claim_id`

## Incremental Watermark

`last_updated`

## Target

`LH_Silver.dbo.silver_claims`

## Production Guarantees

- Incremental processing
- Data-quality enforcement
- Duplicate protection
- Idempotent Delta MERGE
- INSERT and UPDATE support
- Batch reconciliation
- Operational auditability
- Watermark advances only after successful processing
- Restart-safe execution


In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# ============================================================
# STEP 1 - INITIALIZE CLAIMS INCREMENTAL PIPELINE
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

from datetime import datetime
import uuid

# ------------------------------------------------------------
# Pipeline metadata
# ------------------------------------------------------------

PIPELINE_NAME = "PL_Insurance_Medallion_ETL"
SOURCE_NAME   = "CLAIMS"

BATCH_ID = str(uuid.uuid4())
RUN_START_TS = datetime.now()

# ------------------------------------------------------------
# Physical tables
# ------------------------------------------------------------

SOURCE_TABLE  = "LH_Bronze.dbo.bronze_claims"
TARGET_TABLE  = "LH_Silver.dbo.silver_claims"

CONTROL_TABLE = "LH_Silver.dbo.etl_control"
AUDIT_TABLE   = "LH_Silver.dbo.etl_batch_audit"

BUSINESS_KEY     = "claim_id"
WATERMARK_COLUMN = "last_updated"

print("Claims incremental pipeline initialized.")
print("----------------------------------------")
print(f"Pipeline       : {PIPELINE_NAME}")
print(f"Source         : {SOURCE_NAME}")
print(f"Batch ID       : {BATCH_ID}")
print(f"Run start      : {RUN_START_TS}")
print(f"Bronze table   : {SOURCE_TABLE}")
print(f"Silver table   : {TARGET_TABLE}")
print(f"Control table  : {CONTROL_TABLE}")
print(f"Audit table    : {AUDIT_TABLE}")
print(f"Business key   : {BUSINESS_KEY}")
print(f"Watermark      : {WATERMARK_COLUMN}")


StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 3, Finished, Available, Finished, False)

Claims incremental pipeline initialized.
----------------------------------------
Pipeline       : PL_Insurance_Medallion_ETL
Source         : CLAIMS
Batch ID       : 51825150-f8a2-486f-af38-0acb287ea7fd
Run start      : 2026-08-22 12:35:38.579189
Bronze table   : LH_Bronze.dbo.bronze_claims
Silver table   : LH_Silver.dbo.silver_claims
Control table  : LH_Silver.dbo.etl_control
Audit table    : LH_Silver.dbo.etl_batch_audit
Business key   : claim_id
Watermark      : last_updated


## Step 2 — Inspect Bronze Claims Source

Before applying incremental logic, the Bronze Claims source is inspected to confirm:

- Source availability
- Record count
- Physical schema
- Business key (`claim_id`)
- Incremental watermark (`last_updated`)
- Representative source records

The observed Bronze schema will drive validation and Silver transformation rules rather than relying on assumed column definitions.

In [2]:

# ============================================================
# STEP 2 - INSPECT BRONZE CLAIMS SOURCE
# ============================================================

bronze_claims_df = spark.table(SOURCE_TABLE)

bronze_claims_count = bronze_claims_df.count()

print("Bronze Claims source inspected.")
print("----------------------------------------")
print(f"Table          : {SOURCE_TABLE}")
print(f"Bronze rows    : {bronze_claims_count}")
print(f"Business key   : {BUSINESS_KEY}")
print(f"Watermark      : {WATERMARK_COLUMN}")

# ------------------------------------------------------------
# Verify required technical columns exist
# ------------------------------------------------------------

required_columns = [BUSINESS_KEY, WATERMARK_COLUMN]

missing_columns = [
    col_name
    for col_name in required_columns
    if col_name not in bronze_claims_df.columns
]

assert len(missing_columns) == 0, \
    f"Required Claims columns missing: {missing_columns}"

print()
print("Required Claims columns verified.")

# ------------------------------------------------------------
# Inspect schema
# ------------------------------------------------------------

print()
print("Bronze Claims schema:")
bronze_claims_df.printSchema()

# ------------------------------------------------------------
# Inspect sample data
# ------------------------------------------------------------

display(
    bronze_claims_df
        .orderBy(F.col(WATERMARK_COLUMN).desc())
        .limit(10)
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 4, Finished, Available, Finished, False)

Bronze Claims source inspected.
----------------------------------------
Table          : LH_Bronze.dbo.bronze_claims
Bronze rows    : 1200
Business key   : claim_id
Watermark      : last_updated

Required Claims columns verified.

Bronze Claims schema:
root
 |-- claim_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- claim_date: string (nullable = true)
 |-- incident_date: string (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- claim_amount: string (nullable = true)
 |-- approved_amount: string (nullable = true)
 |-- claim_status: string (nullable = true)
 |-- description: string (nullable = true)
 |-- reported_channel: string (nullable = true)
 |-- adjuster_id: string (nullable = true)
 |-- last_updated: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 82d5247f-6df6-43e9-9dfe-c0a4c3a418ce)

## Step 3 — Load Claims ETL Configuration and Watermark

The Claims pipeline uses the centralized ETL control table to determine its
incremental processing state.

This step:

- Reads the active `CLAIMS` configuration.
- Retrieves the configured source and target tables.
- Retrieves the watermark column.
- Retrieves the previously committed watermark.
- Confirms that exactly one active configuration exists.

The stored watermark represents the last successfully processed Claims record.
It must only advance after the complete Bronze-to-Silver transaction succeeds.

In [3]:
# ============================================================
# STEP 3 - LOAD CLAIMS CONFIGURATION AND WATERMARK
# ============================================================

claims_config_df = (
    spark.table(CONTROL_TABLE)
        .filter(
            (F.col("source_name") == SOURCE_NAME) &
            (F.col("is_active") == True)
        )
)

claims_config_count = claims_config_df.count()

assert claims_config_count == 1, \
    f"Expected exactly one active CLAIMS configuration, found {claims_config_count}."

claims_config = claims_config_df.first()

CONFIG_SOURCE_TABLE = claims_config["source_table"]
CONFIG_TARGET_TABLE = claims_config["target_table"]
CONFIG_WATERMARK_COLUMN = claims_config["watermark_column"]
LAST_WATERMARK = claims_config["last_watermark"]
LOAD_TYPE = claims_config["load_type"]

print("Claims ETL configuration loaded.")
print("----------------------------------------")
print(f"Source name      : {SOURCE_NAME}")
print(f"Source table     : {CONFIG_SOURCE_TABLE}")
print(f"Target table     : {CONFIG_TARGET_TABLE}")
print(f"Watermark column : {CONFIG_WATERMARK_COLUMN}")
print(f"Stored watermark : {LAST_WATERMARK}")
print(f"Load type        : {LOAD_TYPE}")

display(claims_config_df)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 5, Finished, Available, Finished, False)

Claims ETL configuration loaded.
----------------------------------------
Source name      : CLAIMS
Source table     : LH_Bronze.dbo.bronze_claims
Target table     : LH_Silver.dbo.silver_claims
Watermark column : last_updated
Stored watermark : 1900-01-01 00:00:00
Load type        : INCREMENTAL


SynapseWidget(Synapse.DataFrame, b278ad8b-02e5-4848-9fdc-b819b06ee0e4)


## Step 4 — Incremental Claims Extraction

Claims are extracted from Bronze using the watermark stored in the centralized
ETL control table.

Only records whose `last_updated` value is greater than the previously committed
watermark are selected.

For the initial Claims execution, the stored watermark is `1900-01-01`, so all
eligible Bronze Claims are expected to enter incremental processing.

The raw Bronze watermark is parsed into a timestamp before comparison to ensure
the incremental filter is type-safe.

The watermark is **not updated at this stage**. It will advance only after
validation, transformation, Delta MERGE, reconciliation, and audit processing
complete successfully.

In [4]:

# ============================================================
# STEP 4 - EXTRACT INCREMENTAL CLAIMS
# ============================================================

# Parse Bronze watermark into timestamp
claims_with_watermark_df = (
    bronze_claims_df
        .withColumn(
            "_watermark_ts",
            F.to_timestamp(F.col(CONFIG_WATERMARK_COLUMN))
        )
)

# Verify watermark parsing
invalid_watermark_count = (
    claims_with_watermark_df
        .filter(
            F.col(CONFIG_WATERMARK_COLUMN).isNotNull() &
            F.col("_watermark_ts").isNull()
        )
        .count()
)

assert invalid_watermark_count == 0, \
    f"Found {invalid_watermark_count} Claims records with invalid watermark values."

# Incremental extraction
incremental_claims_df = (
    claims_with_watermark_df
        .filter(
            F.col("_watermark_ts") > F.lit(LAST_WATERMARK).cast("timestamp")
        )
)

incremental_claims_count = incremental_claims_df.count()

print("Claims incremental extraction completed.")
print("----------------------------------------")
print(f"Bronze rows         : {bronze_claims_count}")
print(f"Stored watermark    : {LAST_WATERMARK}")
print(f"Incremental records : {incremental_claims_count}")
print(f"Invalid watermarks  : {invalid_watermark_count}")

display(
    incremental_claims_df
        .orderBy(F.col("_watermark_ts").desc())
        .limit(10)
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 6, Finished, Available, Finished, False)

Claims incremental extraction completed.
----------------------------------------
Bronze rows         : 1200
Stored watermark    : 1900-01-01 00:00:00
Incremental records : 1200
Invalid watermarks  : 0


SynapseWidget(Synapse.DataFrame, a88c046d-d5cf-44e3-b486-46ffc50cdaa5)


## Step 5 — Claims Data Quality Validation

Incremental Claims records are validated before entering the Silver layer.

### Core validation rules

A Claim is considered valid when:

- `claim_id` is populated.
- `policy_id` is populated.
- `customer_id` is populated.
- `claim_date` can be parsed as a valid date.
- `incident_date` can be parsed as a valid date.
- `claim_amount` can be converted to a numeric value.
- `claim_amount` is non-negative.
- `approved_amount`, when supplied, can be converted to a numeric value.
- `approved_amount` is non-negative when supplied.
- `claim_status` is populated.
- `last_updated` contains a valid timestamp.

`approved_amount` is allowed to be NULL because a Claim may not yet have
an approved settlement amount while it is Submitted or Under Review.

Invalid records are separated from valid records before Silver processing.
This prevents malformed Bronze data from contaminating the curated layer.

In [5]:
# ============================================================
# STEP 5 - VALIDATE INCREMENTAL CLAIMS
# ============================================================

claims_validation_df = (
    incremental_claims_df

    # --------------------------------------------------------
    # Parse raw Bronze values
    # --------------------------------------------------------
    .withColumn(
        "_parsed_claim_date",
        F.to_date(F.col("claim_date"))
    )

    .withColumn(
        "_parsed_incident_date",
        F.to_date(F.col("incident_date"))
    )

    .withColumn(
        "_parsed_claim_amount",
        F.col("claim_amount").cast("double")
    )

    .withColumn(
        "_parsed_approved_amount",
        F.col("approved_amount").cast("double")
    )

    .withColumn(
        "_parsed_last_updated",
        F.to_timestamp(F.col("last_updated"))
    )
)

# ------------------------------------------------------------
# Individual validation rules
# ------------------------------------------------------------

claims_validation_df = (
    claims_validation_df

    .withColumn(
        "_valid_claim_id",
        F.col("claim_id").isNotNull() &
        (F.trim(F.col("claim_id")) != "")
    )

    .withColumn(
        "_valid_policy_id",
        F.col("policy_id").isNotNull() &
        (F.trim(F.col("policy_id")) != "")
    )

    .withColumn(
        "_valid_customer_id",
        F.col("customer_id").isNotNull() &
        (F.trim(F.col("customer_id")) != "")
    )

    .withColumn(
        "_valid_claim_date",
        F.col("_parsed_claim_date").isNotNull()
    )

    .withColumn(
        "_valid_incident_date",
        F.col("_parsed_incident_date").isNotNull()
    )

    .withColumn(
        "_valid_claim_amount",
        F.col("_parsed_claim_amount").isNotNull() &
        (F.col("_parsed_claim_amount") >= 0)
    )

    # approved_amount may legitimately be NULL
    .withColumn(
        "_valid_approved_amount",
        F.col("approved_amount").isNull() |
        (
            F.col("_parsed_approved_amount").isNotNull() &
            (F.col("_parsed_approved_amount") >= 0)
        )
    )

    .withColumn(
        "_valid_claim_status",
        F.col("claim_status").isNotNull() &
        (F.trim(F.col("claim_status")) != "")
    )

    .withColumn(
        "_valid_last_updated",
        F.col("_parsed_last_updated").isNotNull()
    )
)

# ------------------------------------------------------------
# Overall validity
# ------------------------------------------------------------

claims_validation_df = (
    claims_validation_df
    .withColumn(
        "_is_valid",
        F.col("_valid_claim_id") &
        F.col("_valid_policy_id") &
        F.col("_valid_customer_id") &
        F.col("_valid_claim_date") &
        F.col("_valid_incident_date") &
        F.col("_valid_claim_amount") &
        F.col("_valid_approved_amount") &
        F.col("_valid_claim_status") &
        F.col("_valid_last_updated")
    )
)

# ------------------------------------------------------------
# Split valid / rejected records
# ------------------------------------------------------------

valid_claims_df = (
    claims_validation_df
        .filter(F.col("_is_valid"))
)

rejected_claims_df = (
    claims_validation_df
        .filter(~F.col("_is_valid"))
)

valid_claims_count = valid_claims_df.count()
rejected_claims_count = rejected_claims_df.count()

# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

assert (
    valid_claims_count + rejected_claims_count
    == incremental_claims_count
), "Claims validation reconciliation failed."

print("Claims validation completed.")
print("----------------------------------------")
print(f"Incremental records : {incremental_claims_count}")
print(f"Valid records       : {valid_claims_count}")
print(f"Rejected records    : {rejected_claims_count}")
print(
    f"Reconciliation      : "
    f"{valid_claims_count + rejected_claims_count}"
)

display(
    rejected_claims_df.select(
        "claim_id",
        "policy_id",
        "customer_id",
        "claim_date",
        "incident_date",
        "claim_amount",
        "approved_amount",
        "claim_status",
        "last_updated"
    ).limit(20)
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 7, Finished, Available, Finished, False)

Claims validation completed.
----------------------------------------
Incremental records : 1200
Valid records       : 1198
Rejected records    : 2
Reconciliation      : 1200


SynapseWidget(Synapse.DataFrame, 3215c14e-e39f-4c30-bfe1-082bceac2013)

## Step 6 — Deduplicate Valid Claims

Valid Claims are deduplicated using `claim_id` as the business key.

When multiple Bronze versions of the same Claim exist, the record with the most
recent `last_updated` timestamp is retained for Silver processing.

This ensures:

- One incoming record per Claim business key
- Older Claim versions do not overwrite newer information
- Delta MERGE receives a deterministic source dataset
- Duplicate Bronze versions do not create duplicate Silver Claims

Deduplication occurs **after data-quality validation**, ensuring that an invalid
newer version cannot replace a valid Claim record.

In [6]:
# ============================================================
# STEP 6 - DEDUPLICATE VALID CLAIMS
# ============================================================

claim_window = (
    Window
        .partitionBy("claim_id")
        .orderBy(F.col("_parsed_last_updated").desc())
)

ranked_claims_df = (
    valid_claims_df
        .withColumn(
            "_claim_rank",
            F.row_number().over(claim_window)
        )
)

deduplicated_claims_df = (
    ranked_claims_df
        .filter(F.col("_claim_rank") == 1)
)

duplicate_claim_versions_df = (
    ranked_claims_df
        .filter(F.col("_claim_rank") > 1)
)

deduplicated_claims_count = deduplicated_claims_df.count()
duplicate_claim_versions_count = duplicate_claim_versions_df.count()

# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

assert (
    deduplicated_claims_count + duplicate_claim_versions_count
    == valid_claims_count
), "Claims deduplication reconciliation failed."

print("Claims deduplication completed.")
print("----------------------------------------")
print(f"Valid records       : {valid_claims_count}")
print(f"Unique Claims       : {deduplicated_claims_count}")
print(f"Duplicate versions  : {duplicate_claim_versions_count}")
print(
    f"Reconciliation      : "
    f"{deduplicated_claims_count + duplicate_claim_versions_count}"
)

display(
    duplicate_claim_versions_df.select(
        "claim_id",
        "policy_id",
        "customer_id",
        "claim_status",
        "claim_amount",
        "approved_amount",
        "last_updated",
        "_claim_rank"
    ).limit(20)
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 8, Finished, Available, Finished, False)

Claims deduplication completed.
----------------------------------------
Valid records       : 1198
Unique Claims       : 1198
Duplicate versions  : 0
Reconciliation      : 1198


SynapseWidget(Synapse.DataFrame, cba88bc3-43b2-4ca5-8746-b93e8e6870d0)

## Step 7 — Transform Claims into Canonical Silver Schema

Validated and deduplicated Claims are transformed from raw Bronze representation
into the standardized Silver schema.

### Transformations

- Convert `claim_date` and `incident_date` from strings to dates.
- Convert `claim_amount` and `approved_amount` to numeric values.
- Normalize `claim_type`, `claim_status`, and `reported_channel`.
- Preserve Claim relationships to Policy and Customer.
- Convert `last_updated` to a timestamp.
- Add `_silver_processed_ts` for operational lineage.

The Silver layer provides strongly typed, standardized Claims data suitable for
downstream analytics, Gold-layer modeling, and reporting.

In [7]:
# ============================================================
# STEP 7 - TRANSFORM CLAIMS FOR SILVER
# ============================================================

silver_ready_claims_df = (
    deduplicated_claims_df
    .select(
        F.trim(F.col("claim_id"))
            .alias("claim_id"),

        F.trim(F.col("policy_id"))
            .alias("policy_id"),

        F.trim(F.col("customer_id"))
            .alias("customer_id"),

        F.col("_parsed_claim_date")
            .alias("claim_date"),

        F.col("_parsed_incident_date")
            .alias("incident_date"),

        F.upper(F.trim(F.col("claim_type")))
            .alias("claim_type"),

        F.col("_parsed_claim_amount")
            .cast("double")
            .alias("claim_amount"),

        F.col("_parsed_approved_amount")
            .cast("double")
            .alias("approved_amount"),

        F.upper(F.trim(F.col("claim_status")))
            .alias("claim_status"),

        F.trim(F.col("description"))
            .alias("description"),

        F.upper(F.trim(F.col("reported_channel")))
            .alias("reported_channel"),

        F.trim(F.col("adjuster_id"))
            .alias("adjuster_id"),

        F.col("_parsed_last_updated")
            .alias("last_updated"),

        F.current_timestamp()
            .alias("_silver_processed_ts")
    )
)

silver_ready_claims_count = silver_ready_claims_df.count()

assert silver_ready_claims_count == deduplicated_claims_count, \
    "Claims Silver transformation changed the expected record count."

print("Claims Silver transformation completed.")
print("----------------------------------------")
print(f"Unique valid Claims  : {deduplicated_claims_count}")
print(f"Silver-ready Claims  : {silver_ready_claims_count}")

print()
silver_ready_claims_df.printSchema()

display(
    silver_ready_claims_df
        .orderBy(F.col("last_updated").desc())
        .limit(10)
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 9, Finished, Available, Finished, False)

Claims Silver transformation completed.
----------------------------------------
Unique valid Claims  : 1198
Silver-ready Claims  : 1198

root
 |-- claim_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- claim_date: date (nullable = true)
 |-- incident_date: date (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- approved_amount: double (nullable = true)
 |-- claim_status: string (nullable = true)
 |-- description: string (nullable = true)
 |-- reported_channel: string (nullable = true)
 |-- adjuster_id: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, b11a2ca1-7310-4fa5-b51f-9604a3bc890e)


## Step 8 — Inspect Existing Silver Claims

Before performing change detection or Delta MERGE, the existing Silver Claims
target is inspected.

This establishes the current target state and confirms:

- Whether the Silver Claims table already contains data.
- The existing Silver schema.
- Current Silver row count.
- Whether incoming Claims represent INSERT, UPDATE, or NO-OP operations.

No data is modified during this step.

In [8]:

# ============================================================
# STEP 8 - INSPECT EXISTING SILVER CLAIMS
# ============================================================

existing_silver_claims_df = spark.table(CONFIG_TARGET_TABLE)

existing_silver_claims_count = existing_silver_claims_df.count()

print("Existing Silver Claims table inspected.")
print("----------------------------------------")
print(f"Target table      : {CONFIG_TARGET_TABLE}")
print(f"Existing rows     : {existing_silver_claims_count}")

print()
print("Existing Silver Claims schema:")
existing_silver_claims_df.printSchema()

display(
    existing_silver_claims_df
        .orderBy(F.col("last_updated").desc())
        .limit(10)
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 10, Finished, Available, Finished, False)

Existing Silver Claims table inspected.
----------------------------------------
Target table      : LH_Silver.dbo.silver_claims
Existing rows     : 1200

Existing Silver Claims schema:
root
 |-- claim_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- claim_date: date (nullable = true)
 |-- incident_date: date (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- approved_amount: double (nullable = true)
 |-- claim_status: string (nullable = true)
 |-- description: string (nullable = true)
 |-- reported_channel: string (nullable = true)
 |-- adjuster_id: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, 59844f7f-4dc5-4106-943d-615c968d2672)

## Step 9 — Classify Claims as INSERT, UPDATE, or NO-OP

The validated and standardized incremental Claims are compared with the
existing Silver Claims table using `claim_id` as the business key.

Each incoming Claim is classified as:

- **INSERT** — the `claim_id` does not exist in Silver.
- **UPDATE** — the Claim exists, but one or more business attributes changed.
- **NO-OP** — the Claim exists and its business values are unchanged.

Operational metadata such as `_silver_processed_ts` is intentionally excluded
from change detection.

This prevents unnecessary Delta updates and makes repeated pipeline execution
idempotent.

In [9]:
# ============================================================
# STEP 9 - CLASSIFY CLAIMS: INSERT / UPDATE / NO-OP
# ============================================================

src = silver_ready_claims_df.alias("src")
tgt = existing_silver_claims_df.alias("tgt")

matched_claims_df = (
    src.join(
        tgt,
        F.col("src.claim_id") == F.col("tgt.claim_id"),
        "inner"
    )
)

insert_claims_df = (
    src.join(
        tgt,
        F.col("src.claim_id") == F.col("tgt.claim_id"),
        "left_anti"
    )
)

# ------------------------------------------------------------
# Compare BUSINESS columns only
# Null-safe comparison is important for approved_amount,
# description, adjuster_id, etc.
# ------------------------------------------------------------

change_condition = (
    ~F.col("src.policy_id").eqNullSafe(F.col("tgt.policy_id"))
    |
    ~F.col("src.customer_id").eqNullSafe(F.col("tgt.customer_id"))
    |
    ~F.col("src.claim_date").eqNullSafe(F.col("tgt.claim_date"))
    |
    ~F.col("src.incident_date").eqNullSafe(F.col("tgt.incident_date"))
    |
    ~F.col("src.claim_type").eqNullSafe(F.col("tgt.claim_type"))
    |
    ~F.col("src.claim_amount").eqNullSafe(F.col("tgt.claim_amount"))
    |
    ~F.col("src.approved_amount").eqNullSafe(F.col("tgt.approved_amount"))
    |
    ~F.col("src.claim_status").eqNullSafe(F.col("tgt.claim_status"))
    |
    ~F.col("src.description").eqNullSafe(F.col("tgt.description"))
    |
    ~F.col("src.reported_channel").eqNullSafe(F.col("tgt.reported_channel"))
    |
    ~F.col("src.adjuster_id").eqNullSafe(F.col("tgt.adjuster_id"))
    |
    ~F.col("src.last_updated").eqNullSafe(F.col("tgt.last_updated"))
)

changed_claims_joined_df = matched_claims_df.filter(change_condition)

unchanged_claims_joined_df = matched_claims_df.filter(~change_condition)

# ------------------------------------------------------------
# IMPORTANT:
# Return only src.* so later MERGE does not contain duplicate
# claim_id / _silver_processed_ts columns.
# ------------------------------------------------------------

changed_claims_df = changed_claims_joined_df.select("src.*")

unchanged_claims_df = unchanged_claims_joined_df.select("src.*")

# ------------------------------------------------------------
# Counts
# ------------------------------------------------------------

insert_count = insert_claims_df.count()
matched_count = matched_claims_df.count()
update_count = changed_claims_df.count()
noop_count = unchanged_claims_df.count()

# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

assert (
    insert_count + update_count + noop_count
    == silver_ready_claims_count
), "Claims INSERT/UPDATE/NO-OP reconciliation failed."

print("Claims change detection completed.")
print("----------------------------------------")
print(f"Silver-ready Claims : {silver_ready_claims_count}")
print(f"INSERT candidates   : {insert_count}")
print(f"Matched Claims      : {matched_count}")
print(f"UPDATE candidates   : {update_count}")
print(f"NO-OP Claims        : {noop_count}")
print(
    f"Reconciliation      : "
    f"{insert_count + update_count + noop_count}"
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 11, Finished, Available, Finished, False)

Claims change detection completed.
----------------------------------------
Silver-ready Claims : 1198
INSERT candidates   : 0
Matched Claims      : 1198
UPDATE candidates   : 1198
NO-OP Claims        : 0
Reconciliation      : 1198


### Step 9A — Diagnose Claims Change Detection

All incoming Claims matched existing Silver business keys, but every matched
record was classified as an UPDATE.

Before performing a Delta MERGE, the pipeline compares individual business
attributes to determine which columns are causing the differences.

This protects Silver from unnecessary mass updates caused only by differences
in normalization or formatting.

In [10]:
# ============================================================
# STEP 9A - DIAGNOSE CLAIM COLUMN DIFFERENCES
# ============================================================

comparison_columns = [
    "policy_id",
    "customer_id",
    "claim_date",
    "incident_date",
    "claim_type",
    "claim_amount",
    "approved_amount",
    "claim_status",
    "description",
    "reported_channel",
    "adjuster_id",
    "last_updated"
]

difference_counts = []

for column_name in comparison_columns:

    difference_count = (
        matched_claims_df
        .filter(
            ~F.col(f"src.{column_name}")
             .eqNullSafe(F.col(f"tgt.{column_name}"))
        )
        .count()
    )

    difference_counts.append(
        (column_name, difference_count)
    )

difference_counts_df = spark.createDataFrame(
    difference_counts,
    ["column_name", "different_rows"]
)

print("Claims column-level difference analysis completed.")
print("-----------------------------------------------")

display(
    difference_counts_df
        .orderBy(F.col("different_rows").desc())
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 12, Finished, Available, Finished, False)

Claims column-level difference analysis completed.
-----------------------------------------------


SynapseWidget(Synapse.DataFrame, 5cf934af-7613-46b9-8cbf-4b727ba48b8d)

### Step 9B — Canonicalize Existing Silver for Change Detection

Column-level diagnostics identified `reported_channel` as the only difference
between the incoming canonical Claims and the existing Silver Claims.

The existing Silver table contains mixed-case channel values such as `Phone`,
`Mobile`, and `Web`, while the production transformation standardizes these as
`PHONE`, `MOBILE`, and `WEB`.

For change detection, the existing Silver values are therefore normalized using
the same canonical business rules as the incoming dataset.

This prevents formatting-only differences from generating false UPDATEs while
still allowing genuine channel changes to be detected.

In [11]:
# ============================================================
# STEP 9B - CANONICALIZE EXISTING SILVER CLAIMS
# ============================================================

canonical_silver_claims_df = (
    existing_silver_claims_df
    .withColumn(
        "reported_channel",
        F.upper(F.trim(F.col("reported_channel")))
    )
)

channel_difference_after_normalization = (
    silver_ready_claims_df.alias("src")
    .join(
        canonical_silver_claims_df.alias("tgt"),
        F.col("src.claim_id") == F.col("tgt.claim_id"),
        "inner"
    )
    .filter(
        ~F.col("src.reported_channel")
         .eqNullSafe(F.col("tgt.reported_channel"))
    )
    .count()
)

print("Existing Silver Claims canonicalization completed.")
print("------------------------------------------------")
print(
    f"reported_channel differences after normalization : "
    f"{channel_difference_after_normalization}"
)

assert channel_difference_after_normalization == 0, \
    "reported_channel normalization did not resolve the differences."

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 13, Finished, Available, Finished, False)

Existing Silver Claims canonicalization completed.
------------------------------------------------
reported_channel differences after normalization : 0


In [12]:
# ============================================================
# STEP 9C - RECLASSIFY CLAIMS AFTER CANONICALIZATION
# ============================================================

src = silver_ready_claims_df.alias("src")
tgt = canonical_silver_claims_df.alias("tgt")

matched_claims_df = (
    src.join(
        tgt,
        F.col("src.claim_id") == F.col("tgt.claim_id"),
        "inner"
    )
)

insert_claims_df = (
    src.join(
        tgt,
        F.col("src.claim_id") == F.col("tgt.claim_id"),
        "left_anti"
    )
)

change_condition = (
    ~F.col("src.policy_id").eqNullSafe(F.col("tgt.policy_id"))
    |
    ~F.col("src.customer_id").eqNullSafe(F.col("tgt.customer_id"))
    |
    ~F.col("src.claim_date").eqNullSafe(F.col("tgt.claim_date"))
    |
    ~F.col("src.incident_date").eqNullSafe(F.col("tgt.incident_date"))
    |
    ~F.col("src.claim_type").eqNullSafe(F.col("tgt.claim_type"))
    |
    ~F.col("src.claim_amount").eqNullSafe(F.col("tgt.claim_amount"))
    |
    ~F.col("src.approved_amount").eqNullSafe(F.col("tgt.approved_amount"))
    |
    ~F.col("src.claim_status").eqNullSafe(F.col("tgt.claim_status"))
    |
    ~F.col("src.description").eqNullSafe(F.col("tgt.description"))
    |
    ~F.col("src.reported_channel").eqNullSafe(F.col("tgt.reported_channel"))
    |
    ~F.col("src.adjuster_id").eqNullSafe(F.col("tgt.adjuster_id"))
    |
    ~F.col("src.last_updated").eqNullSafe(F.col("tgt.last_updated"))
)

changed_claims_df = (
    matched_claims_df
    .filter(change_condition)
    .select("src.*")
)

unchanged_claims_df = (
    matched_claims_df
    .filter(~change_condition)
    .select("src.*")
)

insert_count = insert_claims_df.count()
update_count = changed_claims_df.count()
noop_count = unchanged_claims_df.count()

assert (
    insert_count + update_count + noop_count
    == silver_ready_claims_count
), "Claims reclassification reconciliation failed."

print("Claims canonical change detection completed.")
print("--------------------------------------------")
print(f"Silver-ready Claims : {silver_ready_claims_count}")
print(f"INSERT candidates   : {insert_count}")
print(f"UPDATE candidates   : {update_count}")
print(f"NO-OP Claims        : {noop_count}")
print(f"Reconciliation      : {insert_count + update_count + noop_count}")

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 14, Finished, Available, Finished, False)

Claims canonical change detection completed.
--------------------------------------------
Silver-ready Claims : 1198
INSERT candidates   : 0
UPDATE candidates   : 0
NO-OP Claims        : 1198
Reconciliation      : 1198



## Step 10 — Complete Initial Claims Incremental Run

Change detection confirmed that all 1,198 valid incoming Claims already exist
in Silver with equivalent canonical business values.

Therefore, no Delta MERGE is required for this execution.

The pipeline will:

1. Skip unnecessary Silver writes.
2. Record the execution in the ETL batch audit table.
3. Record the two rejected Bronze Claims.
4. Advance the Claims watermark to the maximum successfully processed
   `last_updated` timestamp.
5. Verify the persisted audit and control state.

This establishes the restart point for future Claims incremental processing.

In [14]:
# ============================================================
# STEP 10A - DEFINE CLAIMS AUDIT SCHEMA
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("source_count", LongType(), False),
    StructField("insert_count", LongType(), False),
    StructField("update_count", LongType(), False),
    StructField("reject_count", LongType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

print("Claims audit schema initialized.")
audit_schema.simpleString()

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 16, Finished, Available, Finished, False)

Claims audit schema initialized.


'struct<batch_id:string,pipeline_name:string,table_name:string,start_time:timestamp,end_time:timestamp,source_count:bigint,insert_count:bigint,update_count:bigint,reject_count:bigint,status:string,error_message:string>'

In [15]:
# ============================================================
# STEP 10 - COMPLETE INITIAL CLAIMS INCREMENTAL RUN
# ============================================================

from datetime import datetime
from pyspark.sql import Row

# ------------------------------------------------------------
# 1. Calculate new watermark from VALID processed Claims
# ------------------------------------------------------------

NEW_CLAIMS_WATERMARK = (
    silver_ready_claims_df
    .agg(F.max("last_updated").alias("new_watermark"))
    .first()["new_watermark"]
)

assert NEW_CLAIMS_WATERMARK is not None, \
    "Claims watermark calculation returned NULL."

assert NEW_CLAIMS_WATERMARK >= LAST_WATERMARK, \
    "New Claims watermark cannot be earlier than stored watermark."

print("Claims watermark calculated.")
print("----------------------------------------")
print(f"Previous watermark : {LAST_WATERMARK}")
print(f"New watermark      : {NEW_CLAIMS_WATERMARK}")

# ------------------------------------------------------------
# 2. MERGE decision
# ------------------------------------------------------------

merge_source_count = insert_count + update_count

if merge_source_count > 0:
    raise Exception(
        "Unexpected Claims changes detected during initial NO-OP run."
    )

print()
print("No Claims INSERTs or UPDATEs detected.")
print("Delta MERGE skipped.")

# ------------------------------------------------------------
# 3. Write audit record
# ------------------------------------------------------------

RUN_END_TS = datetime.now()

# ------------------------------------------------------------
# Audit schema
# Keep NB_04 self-contained
# ------------------------------------------------------------

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("source_count", LongType(), False),
    StructField("insert_count", LongType(), False),
    StructField("update_count", LongType(), False),
    StructField("reject_count", LongType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

claims_audit_record = [
    Row(
        batch_id=BATCH_ID,
        pipeline_name=PIPELINE_NAME,
        table_name=SOURCE_NAME,
        start_time=RUN_START_TS,
        end_time=RUN_END_TS,
        source_count=incremental_claims_count,
        insert_count=insert_count,
        update_count=update_count,
        reject_count=rejected_claims_count,
        status="SUCCESS",
        error_message=None
    )
]

claims_audit_df = spark.createDataFrame(
    claims_audit_record,
    schema=audit_schema
)

(
    claims_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print()
print("Claims audit record written.")

# ------------------------------------------------------------
# 4. Update Claims watermark
# ------------------------------------------------------------

spark.sql(f"""
    UPDATE {CONTROL_TABLE}
       SET last_watermark = TIMESTAMP('{NEW_CLAIMS_WATERMARK}'),
           _updated_ts = current_timestamp()
     WHERE source_name = '{SOURCE_NAME}'
       AND is_active = true
""")

print("Claims watermark updated.")

# ------------------------------------------------------------
# 5. Verify persisted state
# ------------------------------------------------------------

claims_audit_check_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

claims_control_check_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        &
        (F.col("is_active") == True)
    )
)

audit_row_count = claims_audit_check_df.count()

assert audit_row_count == 1, \
    f"Expected exactly one Claims audit row, found {audit_row_count}."

stored_claims_control = claims_control_check_df.first()

assert stored_claims_control["last_watermark"] == NEW_CLAIMS_WATERMARK, \
    "Claims watermark persistence verification failed."

print()
print("==========================================")
print(" CLAIMS INITIAL INCREMENTAL RUN PASSED")
print("==========================================")
print(f"Bronze incremental rows : {incremental_claims_count}")
print(f"Valid Claims            : {valid_claims_count}")
print(f"Rejected Claims         : {rejected_claims_count}")
print(f"Processed INSERTS       : {insert_count}")
print(f"Processed UPDATES       : {update_count}")
print(f"NO-OP Claims            : {noop_count}")
print(f"Final watermark         : {NEW_CLAIMS_WATERMARK}")

display(claims_audit_check_df)
display(claims_control_check_df)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 17, Finished, Available, Finished, False)

Claims watermark calculated.
----------------------------------------
Previous watermark : 1900-01-01 00:00:00
New watermark      : 2027-10-21 00:00:00

No Claims INSERTs or UPDATEs detected.
Delta MERGE skipped.

Claims audit record written.
Claims watermark updated.

 CLAIMS INITIAL INCREMENTAL RUN PASSED
Bronze incremental rows : 1200
Valid Claims            : 1198
Rejected Claims         : 2
Processed INSERTS       : 0
Processed UPDATES       : 0
NO-OP Claims            : 1198
Final watermark         : 2027-10-21 00:00:00


SynapseWidget(Synapse.DataFrame, f2be83e3-327d-4384-bc67-a727eb59c5ee)

SynapseWidget(Synapse.DataFrame, 4f1fedf1-ff19-458d-a1c7-01c0efe10265)


## Step 11 — Claims Restart / Idempotency Test

This test validates restart-safe incremental processing.

The Claims watermark is now `2027-10-21`. Without adding any new Bronze
records, the incremental extraction is executed again.

### Expected Result

- Bronze Claims remain unchanged.
- Stored watermark remains `2027-10-21`.
- Incremental record count = **0**.
- No Silver MERGE is required.
- No duplicate Claims are created.

This demonstrates that a successful batch can be safely rerun without
reprocessing previously completed records.

In [16]:
# ============================================================
# STEP 11 - CLAIMS RESTART / IDEMPOTENCY TEST
# ============================================================

# Reload persisted control state
restart_control = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        &
        (F.col("is_active") == True)
    )
    .first()
)

RESTART_WATERMARK = restart_control["last_watermark"]

# Reload Bronze
restart_bronze_claims_df = spark.table(CONFIG_SOURCE_TABLE)

restart_bronze_count = restart_bronze_claims_df.count()

# Apply same watermark logic
restart_incremental_claims_df = (
    restart_bronze_claims_df
    .withColumn(
        "_restart_watermark_ts",
        F.to_timestamp(F.col(CONFIG_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_restart_watermark_ts") > F.lit(RESTART_WATERMARK)
    )
)

restart_incremental_count = restart_incremental_claims_df.count()

print("Claims restart test completed.")
print("---------------------------------------")
print(f"Bronze Claims rows : {restart_bronze_count}")
print(f"Stored watermark   : {RESTART_WATERMARK}")
print(f"Incremental rows   : {restart_incremental_count}")

assert restart_incremental_count == 0, \
    f"Expected 0 Claims after restart, found {restart_incremental_count}."

print()
print("CLAIMS RESTART / IDEMPOTENCY TEST PASSED.")

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 18, Finished, Available, Finished, False)

Claims restart test completed.
---------------------------------------
Bronze Claims rows : 1200
Stored watermark   : 2027-10-21 00:00:00
Incremental rows   : 0

CLAIMS RESTART / IDEMPOTENCY TEST PASSED.



## Step 12 — Controlled Claims INSERT + UPDATE Production Test

The base Claims incremental pipeline and restart/idempotency behavior have
been validated.

This controlled test now verifies both Delta MERGE paths in a single batch:

1. **INSERT** — a new Claim that does not currently exist in Silver.
2. **UPDATE** — a newer version of an existing Claim with a genuine business
   attribute change.

The test records will use `last_updated` values later than the current
Claims watermark (`2027-10-21`) so they are detected by the incremental
pipeline.

After processing, the pipeline must produce exactly:

- 1 INSERT
- 1 UPDATE
- 0 rejects
- 2 incremental records

The final restart test must again return zero incremental records.

In [17]:
# ============================================================
# STEP 12A - SELECT EXISTING CLAIM FOR CONTROLLED UPDATE
# ============================================================

TEST_UPDATE_CLAIM_ID = "CLM0000859"

existing_test_claim_df = (
    spark.table(CONFIG_TARGET_TABLE)
    .filter(F.col("claim_id") == TEST_UPDATE_CLAIM_ID)
    .limit(1)
)

existing_test_claim_count = existing_test_claim_df.count()

assert existing_test_claim_count == 1, \
    f"Expected Claim {TEST_UPDATE_CLAIM_ID} to exist in Silver."

print("Existing Claim selected for controlled UPDATE.")
print("-----------------------------------------------")
print(f"Claim ID : {TEST_UPDATE_CLAIM_ID}")

display(
    existing_test_claim_df.select(
        "claim_id",
        "policy_id",
        "customer_id",
        "claim_type",
        "claim_amount",
        "approved_amount",
        "claim_status",
        "reported_channel",
        "last_updated"
    )
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 19, Finished, Available, Finished, False)

Existing Claim selected for controlled UPDATE.
-----------------------------------------------
Claim ID : CLM0000859


SynapseWidget(Synapse.DataFrame, 3ae48ed5-9605-453a-b2c5-9a7c66a88c66)

### Step 12B — Construct Controlled Claims Test Records

Two controlled Bronze records are prepared:

**UPDATE test**
- Existing Claim: `CLM0000859`
- Approved amount: `62654.21 → 65000.00`
- Last updated: `2027-10-22`

**INSERT test**
- New Claim: `CLM999901`
- Last updated: `2027-10-23`

Both records are later than the current Claims watermark and should therefore
be detected during the next incremental execution.

The records are constructed using the existing Bronze schema to avoid datatype
or Delta schema compatibility problems.

In [18]:
# ============================================================
# STEP 12B - CONSTRUCT CONTROLLED INSERT + UPDATE RECORDS
# ============================================================

BRONZE_CLAIMS_TABLE = CONFIG_SOURCE_TABLE

TEST_INSERT_CLAIM_ID = "CLM999901"

TEST_UPDATE_APPROVED_AMOUNT = "65000.00"
TEST_UPDATE_LAST_UPDATED = "2027-10-22"
TEST_INSERT_LAST_UPDATED = "2027-10-23"


# ------------------------------------------------------------
# 1. UPDATE record
# Start from the existing Bronze version so schema stays exact
# ------------------------------------------------------------

controlled_update_claim_df = (
    spark.table(BRONZE_CLAIMS_TABLE)
    .filter(F.col("claim_id") == TEST_UPDATE_CLAIM_ID)
    .orderBy(F.to_timestamp("last_updated").desc())
    .limit(1)
    .withColumn(
        "approved_amount",
        F.lit(TEST_UPDATE_APPROVED_AMOUNT)
    )
    .withColumn(
        "last_updated",
        F.lit(TEST_UPDATE_LAST_UPDATED)
    )
)

assert controlled_update_claim_df.count() == 1, \
    "Could not construct controlled Claims UPDATE record."


# ------------------------------------------------------------
# 2. INSERT record
# Clone an existing valid Bronze record, then replace IDs
# and test-specific business values.
# ------------------------------------------------------------

controlled_insert_claim_df = (
    spark.table(BRONZE_CLAIMS_TABLE)
    .filter(F.col("claim_id") == TEST_UPDATE_CLAIM_ID)
    .orderBy(F.to_timestamp("last_updated").desc())
    .limit(1)

    .withColumn("claim_id", F.lit(TEST_INSERT_CLAIM_ID))
    .withColumn("claim_type", F.lit("COLLISION"))
    .withColumn("claim_amount", F.lit("15000.00"))
    .withColumn("approved_amount", F.lit("12000.00"))
    .withColumn("claim_status", F.lit("APPROVED"))
    .withColumn("reported_channel", F.lit("Web"))
    .withColumn("description", F.lit("Controlled incremental INSERT test"))
    .withColumn("last_updated", F.lit(TEST_INSERT_LAST_UPDATED))
)


# ------------------------------------------------------------
# 3. Combine test records
# ------------------------------------------------------------

controlled_claims_test_df = (
    controlled_update_claim_df
    .unionByName(controlled_insert_claim_df)
)

controlled_test_count = controlled_claims_test_df.count()

assert controlled_test_count == 2, \
    f"Expected 2 controlled Claims records, found {controlled_test_count}."


print("Controlled Claims test records created.")
print("-----------------------------------------")
print(f"UPDATE Claim : {TEST_UPDATE_CLAIM_ID}")
print(f"INSERT Claim : {TEST_INSERT_CLAIM_ID}")
print(f"Test records : {controlled_test_count}")
print()
print("Nothing has been written to Bronze yet.")

display(
    controlled_claims_test_df.select(
        "claim_id",
        "policy_id",
        "customer_id",
        "claim_type",
        "claim_amount",
        "approved_amount",
        "claim_status",
        "reported_channel",
        "last_updated"
    )
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 20, Finished, Available, Finished, False)

Controlled Claims test records created.
-----------------------------------------
UPDATE Claim : CLM0000859
INSERT Claim : CLM999901
Test records : 2

Nothing has been written to Bronze yet.


SynapseWidget(Synapse.DataFrame, 13b9e8f1-1852-4f1e-a9be-71c565fc39a7)


### Step 12C — Append Controlled Claims Records to Bronze

The two validated controlled test records are now appended to the Bronze Claims
Delta table.

Expected Bronze state:

- Previous Bronze rows: **1200**
- Controlled records appended: **2**
- New Bronze rows: **1202**

The records represent:

- `CLM0000859` — newer version of an existing Claim (**UPDATE candidate**)
- `CLM999901` — completely new Claim (**INSERT candidate**)

The Claims watermark remains `2027-10-21` at this point. It will not advance
until the incremental batch completes successfully.

In [19]:

# ============================================================
# STEP 12C - APPEND CONTROLLED CLAIMS TO BRONZE
# ============================================================

bronze_count_before_test = (
    spark.table(BRONZE_CLAIMS_TABLE).count()
)

# Safety check:
# Ensure the controlled INSERT record does not already exist.
existing_insert_test_count = (
    spark.table(BRONZE_CLAIMS_TABLE)
    .filter(F.col("claim_id") == TEST_INSERT_CLAIM_ID)
    .count()
)

assert existing_insert_test_count == 0, \
    f"{TEST_INSERT_CLAIM_ID} already exists in Bronze. Do NOT append again."


# Append using exact Bronze schema
(
    controlled_claims_test_df
    .select(
        spark.table(BRONZE_CLAIMS_TABLE).columns
    )
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_CLAIMS_TABLE)
)


# ------------------------------------------------------------
# Verify Bronze
# ------------------------------------------------------------

bronze_count_after_test = (
    spark.table(BRONZE_CLAIMS_TABLE).count()
)

test_bronze_records_df = (
    spark.table(BRONZE_CLAIMS_TABLE)
    .filter(
        F.col("claim_id").isin(
            TEST_UPDATE_CLAIM_ID,
            TEST_INSERT_CLAIM_ID
        )
    )
    .orderBy(
        "claim_id",
        F.to_timestamp("last_updated").desc()
    )
)

assert bronze_count_after_test == bronze_count_before_test + 2, \
    "Bronze Claims did not increase by exactly two rows."

assert (
    spark.table(BRONZE_CLAIMS_TABLE)
    .filter(F.col("claim_id") == TEST_INSERT_CLAIM_ID)
    .count()
) == 1, \
    "Expected exactly one controlled INSERT Claim in Bronze."


print("Controlled Claims records appended to Bronze.")
print("---------------------------------------------")
print(f"Bronze rows before : {bronze_count_before_test}")
print(f"Bronze rows after  : {bronze_count_after_test}")
print(f"Net increase       : {bronze_count_after_test - bronze_count_before_test}")
print()
print(f"UPDATE test Claim  : {TEST_UPDATE_CLAIM_ID}")
print(f"INSERT test Claim  : {TEST_INSERT_CLAIM_ID}")

display(
    test_bronze_records_df.select(
        "claim_id",
        "claim_type",
        "claim_amount",
        "approved_amount",
        "claim_status",
        "reported_channel",
        "last_updated"
    )
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 21, Finished, Available, Finished, False)

Controlled Claims records appended to Bronze.
---------------------------------------------
Bronze rows before : 1200
Bronze rows after  : 1202
Net increase       : 2

UPDATE test Claim  : CLM0000859
INSERT test Claim  : CLM999901


SynapseWidget(Synapse.DataFrame, 41f97054-1dd6-4cad-922f-9d2b9a8b97a8)

### Step 12D — Process and Classify Controlled Claims Incremental Batch

The controlled Claims records have now been appended to Bronze.

This step executes the production incremental processing logic up to, but **not including, the Delta MERGE**.

The pipeline will:

1. Reload the persisted Claims watermark.
2. Extract Bronze records newer than the watermark.
3. Validate the incremental Claims.
4. Reject invalid records.
5. Deduplicate by `claim_id`, retaining the latest version.
6. Apply canonical Silver transformations.
7. Compare incoming Claims with the existing Silver table.
8. Classify each Claim as:
   - **INSERT** — Claim does not exist in Silver.
   - **UPDATE** — Claim exists but contains a genuine business change.
   - **NO-OP** — Claim exists and contains no business change.
9. Reconcile all classified records before allowing the Delta MERGE.

#### Expected Controlled Test Result

The two Bronze test records should produce:

- Incremental records: **2**
- Valid records: **2**
- Rejected records: **0**
- INSERT candidates: **1**
- UPDATE candidates: **1**
- NO-OP Claims: **0**

Expected classification:

- `CLM999901` → **INSERT**
- `CLM0000859` → **UPDATE**
  - `approved_amount`: **62654.21 → 65000.00**
  - `last_updated`: **2027-10-21 → 2027-10-22**

The Delta MERGE is intentionally **not executed in this step**.

The classification results must pass reconciliation before Silver is modified.


In [20]:
# ============================================================
# STEP 12D - PROCESS CONTROLLED CLAIMS INCREMENTAL BATCH
# Extract -> Validate -> Transform -> Classify
# ============================================================

# ------------------------------------------------------------
# 1. Reload current watermark
# ------------------------------------------------------------

controlled_control_row = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
    .first()
)

CONTROLLED_START_WATERMARK = controlled_control_row["last_watermark"]

print("Controlled Claims batch started.")
print("-----------------------------------------")
print(f"Stored watermark : {CONTROLLED_START_WATERMARK}")


# ------------------------------------------------------------
# 2. Incremental extraction
# ------------------------------------------------------------

controlled_incremental_df = (
    spark.table(CONFIG_SOURCE_TABLE)
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(CONFIG_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") >
        F.lit(CONTROLLED_START_WATERMARK)
    )
)

controlled_incremental_count = controlled_incremental_df.count()

assert controlled_incremental_count == 2, \
    f"Expected exactly 2 incremental Claims, found {controlled_incremental_count}."


# ------------------------------------------------------------
# 3. Validate
# ------------------------------------------------------------

controlled_valid_df = (
    controlled_incremental_df
    .filter(
        F.col("claim_id").isNotNull() &
        F.col("policy_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.to_date("claim_date").isNotNull() &
        F.to_date("incident_date").isNotNull() &
        (F.col("claim_amount").cast("double") >= 0) &
        (
            F.col("approved_amount").isNull() |
            (F.col("approved_amount").cast("double") >= 0)
        ) &
        F.to_timestamp("last_updated").isNotNull()
    )
)

controlled_valid_count = controlled_valid_df.count()
controlled_reject_count = (
    controlled_incremental_count - controlled_valid_count
)

assert controlled_valid_count == 2, \
    f"Expected 2 valid controlled Claims, found {controlled_valid_count}."

assert controlled_reject_count == 0, \
    f"Expected 0 controlled rejects, found {controlled_reject_count}."


# ------------------------------------------------------------
# 4. Deduplicate by Claim ID
# ------------------------------------------------------------

from pyspark.sql.window import Window

controlled_window = (
    Window
    .partitionBy("claim_id")
    .orderBy(F.to_timestamp("last_updated").desc())
)

controlled_unique_df = (
    controlled_valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(controlled_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number", "_watermark_ts")
)


# ------------------------------------------------------------
# 5. Canonical Silver transformation
# ------------------------------------------------------------

controlled_silver_ready_df = (
    controlled_unique_df

    .withColumn("claim_date", F.to_date("claim_date"))
    .withColumn("incident_date", F.to_date("incident_date"))

    .withColumn(
        "claim_type",
        F.upper(F.trim(F.col("claim_type")))
    )

    .withColumn(
        "claim_amount",
        F.col("claim_amount").cast("double")
    )

    .withColumn(
        "approved_amount",
        F.col("approved_amount").cast("double")
    )

    .withColumn(
        "claim_status",
        F.upper(F.trim(F.col("claim_status")))
    )

    .withColumn(
        "reported_channel",
        F.upper(F.trim(F.col("reported_channel")))
    )

    .withColumn(
        "last_updated",
        F.to_timestamp("last_updated")
    )

    .withColumn(
        "_silver_processed_ts",
        F.current_timestamp()
    )
)


# ------------------------------------------------------------
# 6. Canonicalize existing Silver for comparison
# ------------------------------------------------------------

controlled_target_df = (
    spark.table(CONFIG_TARGET_TABLE)

    .withColumn(
        "claim_type",
        F.upper(F.trim(F.col("claim_type")))
    )

    .withColumn(
        "claim_status",
        F.upper(F.trim(F.col("claim_status")))
    )

    .withColumn(
        "reported_channel",
        F.upper(F.trim(F.col("reported_channel")))
    )
)


# ------------------------------------------------------------
# 7. INSERT detection
# ------------------------------------------------------------

controlled_insert_df = (
    controlled_silver_ready_df.alias("src")
    .join(
        controlled_target_df.alias("tgt"),
        F.col("src.claim_id") == F.col("tgt.claim_id"),
        "left_anti"
    )
)


# ------------------------------------------------------------
# 8. Matched Claims
# ------------------------------------------------------------

controlled_matched_df = (
    controlled_silver_ready_df.alias("src")
    .join(
        controlled_target_df.alias("tgt"),
        F.col("src.claim_id") == F.col("tgt.claim_id"),
        "inner"
    )
)


# ------------------------------------------------------------
# 9. Genuine business change detection
# ------------------------------------------------------------

controlled_change_condition = (
    ~F.col("src.policy_id").eqNullSafe(F.col("tgt.policy_id"))
    |
    ~F.col("src.customer_id").eqNullSafe(F.col("tgt.customer_id"))
    |
    ~F.col("src.claim_date").eqNullSafe(F.col("tgt.claim_date"))
    |
    ~F.col("src.incident_date").eqNullSafe(F.col("tgt.incident_date"))
    |
    ~F.col("src.claim_type").eqNullSafe(F.col("tgt.claim_type"))
    |
    ~F.col("src.claim_amount").eqNullSafe(F.col("tgt.claim_amount"))
    |
    ~F.col("src.approved_amount").eqNullSafe(F.col("tgt.approved_amount"))
    |
    ~F.col("src.claim_status").eqNullSafe(F.col("tgt.claim_status"))
    |
    ~F.col("src.description").eqNullSafe(F.col("tgt.description"))
    |
    ~F.col("src.reported_channel").eqNullSafe(F.col("tgt.reported_channel"))
    |
    ~F.col("src.adjuster_id").eqNullSafe(F.col("tgt.adjuster_id"))
    |
    ~F.col("src.last_updated").eqNullSafe(F.col("tgt.last_updated"))
)

controlled_update_df = (
    controlled_matched_df
    .filter(controlled_change_condition)
    .select("src.*")
)

controlled_noop_df = (
    controlled_matched_df
    .filter(~controlled_change_condition)
    .select("src.*")
)


# ------------------------------------------------------------
# 10. Counts + reconciliation
# ------------------------------------------------------------

controlled_insert_count = controlled_insert_df.count()
controlled_update_count = controlled_update_df.count()
controlled_noop_count = controlled_noop_df.count()

assert controlled_insert_count == 1, \
    f"Expected 1 INSERT, found {controlled_insert_count}."

assert controlled_update_count == 1, \
    f"Expected 1 UPDATE, found {controlled_update_count}."

assert controlled_noop_count == 0, \
    f"Expected 0 NO-OP Claims, found {controlled_noop_count}."

assert (
    controlled_insert_count +
    controlled_update_count +
    controlled_noop_count
) == controlled_valid_count, \
    "Controlled Claims reconciliation failed."


print()
print("==========================================")
print(" CONTROLLED CLAIMS CLASSIFICATION PASSED")
print("==========================================")
print(f"Incremental records : {controlled_incremental_count}")
print(f"Valid records       : {controlled_valid_count}")
print(f"Rejected records    : {controlled_reject_count}")
print(f"INSERT candidates   : {controlled_insert_count}")
print(f"UPDATE candidates   : {controlled_update_count}")
print(f"NO-OP Claims        : {controlled_noop_count}")

print()
print("MERGE HAS NOT BEEN EXECUTED YET.")

display(
    controlled_silver_ready_df.select(
        "claim_id",
        "claim_type",
        "claim_amount",
        "approved_amount",
        "claim_status",
        "reported_channel",
        "last_updated"
    )
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 22, Finished, Available, Finished, False)

Controlled Claims batch started.
-----------------------------------------
Stored watermark : 2027-10-21 00:00:00

 CONTROLLED CLAIMS CLASSIFICATION PASSED
Incremental records : 2
Valid records       : 2
Rejected records    : 0
INSERT candidates   : 1
UPDATE candidates   : 1
NO-OP Claims        : 0

MERGE HAS NOT BEEN EXECUTED YET.


SynapseWidget(Synapse.DataFrame, 62686a60-792d-4a4a-87d2-349ee506b8e8)


### Step 12E — Merge Controlled Claims into Silver

The controlled incremental batch has successfully passed extraction,
validation, transformation, classification, and reconciliation.

The batch contains:

- **1 INSERT** — `CLM999901`
- **1 UPDATE** — `CLM0000859`
- **0 rejects**
- **0 NO-OP records**

This step performs the production Delta Lake `MERGE` into the Silver Claims
table using `claim_id` as the business key.

#### MERGE Behavior

When the incoming `claim_id` already exists:

- Update the existing Silver Claim with the latest canonical values.
- Update `last_updated`.
- Refresh `_silver_processed_ts`.

When the incoming `claim_id` does not exist:

- Insert the complete canonical Claim into Silver.

#### Expected Result

Before MERGE:

- Silver Claims rows: **1200**

After MERGE:

- Silver Claims rows: **1201**
- Net row increase: **1**
- `CLM0000859` occurs exactly once and has `approved_amount = 65000.00`
- `CLM999901` occurs exactly once

The UPDATE must modify the existing Silver row rather than create a duplicate.


In [21]:

# ============================================================
# STEP 12E - DELTA MERGE CONTROLLED CLAIMS INTO SILVER
# ============================================================

from delta.tables import DeltaTable

# ------------------------------------------------------------
# 1. Capture Silver state before MERGE
# ------------------------------------------------------------

silver_count_before_controlled_merge = (
    spark.table(CONFIG_TARGET_TABLE).count()
)

print("Preparing controlled Claims Delta MERGE.")
print("-----------------------------------------")
print(f"Silver rows before : {silver_count_before_controlled_merge}")
print(f"INSERT records     : {controlled_insert_count}")
print(f"UPDATE records     : {controlled_update_count}")


# ------------------------------------------------------------
# 2. Build MERGE source
# Only INSERT + UPDATE records are allowed into MERGE
# ------------------------------------------------------------

controlled_merge_source_df = (
    controlled_insert_df
    .unionByName(controlled_update_df)
)

controlled_merge_source_count = controlled_merge_source_df.count()

assert controlled_merge_source_count == 2, \
    f"Expected 2 MERGE records, found {controlled_merge_source_count}."


# ------------------------------------------------------------
# 3. Execute Delta MERGE
# ------------------------------------------------------------

silver_claims_delta = DeltaTable.forName(
    spark,
    CONFIG_TARGET_TABLE
)

(
    silver_claims_delta.alias("tgt")
    .merge(
        controlled_merge_source_df.alias("src"),
        "tgt.claim_id = src.claim_id"
    )

    .whenMatchedUpdate(
        set={
            "policy_id": "src.policy_id",
            "customer_id": "src.customer_id",
            "claim_date": "src.claim_date",
            "incident_date": "src.incident_date",
            "claim_type": "src.claim_type",
            "claim_amount": "src.claim_amount",
            "approved_amount": "src.approved_amount",
            "claim_status": "src.claim_status",
            "description": "src.description",
            "reported_channel": "src.reported_channel",
            "adjuster_id": "src.adjuster_id",
            "last_updated": "src.last_updated",
            "_silver_processed_ts": "src._silver_processed_ts"
        }
    )

    .whenNotMatchedInsert(
        values={
            "claim_id": "src.claim_id",
            "policy_id": "src.policy_id",
            "customer_id": "src.customer_id",
            "claim_date": "src.claim_date",
            "incident_date": "src.incident_date",
            "claim_type": "src.claim_type",
            "claim_amount": "src.claim_amount",
            "approved_amount": "src.approved_amount",
            "claim_status": "src.claim_status",
            "description": "src.description",
            "reported_channel": "src.reported_channel",
            "adjuster_id": "src.adjuster_id",
            "last_updated": "src.last_updated",
            "_silver_processed_ts": "src._silver_processed_ts"
        }
    )

    .execute()
)

print()
print("Controlled Claims Delta MERGE completed.")


# ------------------------------------------------------------
# 4. Verify Silver state
# ------------------------------------------------------------

silver_count_after_controlled_merge = (
    spark.table(CONFIG_TARGET_TABLE).count()
)

controlled_silver_check_df = (
    spark.table(CONFIG_TARGET_TABLE)
    .filter(
        F.col("claim_id").isin(
            TEST_UPDATE_CLAIM_ID,
            TEST_INSERT_CLAIM_ID
        )
    )
    .orderBy("claim_id")
)

update_claim_occurrences = (
    controlled_silver_check_df
    .filter(F.col("claim_id") == TEST_UPDATE_CLAIM_ID)
    .count()
)

insert_claim_occurrences = (
    controlled_silver_check_df
    .filter(F.col("claim_id") == TEST_INSERT_CLAIM_ID)
    .count()
)

updated_approved_amount = (
    controlled_silver_check_df
    .filter(F.col("claim_id") == TEST_UPDATE_CLAIM_ID)
    .select("approved_amount")
    .first()["approved_amount"]
)


# ------------------------------------------------------------
# 5. Assertions
# ------------------------------------------------------------

assert (
    silver_count_after_controlled_merge
    == silver_count_before_controlled_merge + 1
), "Expected Silver Claims to increase by exactly one row."

assert update_claim_occurrences == 1, \
    "UPDATE Claim must occur exactly once in Silver."

assert insert_claim_occurrences == 1, \
    "INSERT Claim must occur exactly once in Silver."

assert abs(float(updated_approved_amount) - 65000.00) < 0.001, \
    "Controlled Claim UPDATE amount was not persisted correctly."


print()
print("==========================================")
print(" CONTROLLED CLAIMS DELTA MERGE PASSED")
print("==========================================")
print(f"Silver rows before : {silver_count_before_controlled_merge}")
print(f"Silver rows after  : {silver_count_after_controlled_merge}")
print(
    f"Net row increase   : "
    f"{silver_count_after_controlled_merge - silver_count_before_controlled_merge}"
)
print(f"Processed INSERTS  : {controlled_insert_count}")
print(f"Processed UPDATES  : {controlled_update_count}")

display(
    controlled_silver_check_df.select(
        "claim_id",
        "policy_id",
        "claim_type",
        "claim_amount",
        "approved_amount",
        "claim_status",
        "reported_channel",
        "last_updated",
        "_silver_processed_ts"
    )
)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 23, Finished, Available, Finished, False)

Preparing controlled Claims Delta MERGE.
-----------------------------------------
Silver rows before : 1200
INSERT records     : 1
UPDATE records     : 1

Controlled Claims Delta MERGE completed.

 CONTROLLED CLAIMS DELTA MERGE PASSED
Silver rows before : 1200
Silver rows after  : 1201
Net row increase   : 1
Processed INSERTS  : 1
Processed UPDATES  : 1


SynapseWidget(Synapse.DataFrame, fe1d66fd-0caf-401b-8a5e-2440f06f1f5b)

### Step 12F — Commit Claims Audit and Watermark

The controlled Claims Delta MERGE completed successfully.

This step finalizes the incremental batch by persisting its operational state.

The pipeline will:

1. Calculate the maximum successfully processed `last_updated` timestamp.
2. Write a SUCCESS record to the ETL batch audit table.
3. Record:
   - Source records: **2**
   - INSERT records: **1**
   - UPDATE records: **1**
   - Rejected records: **0**
4. Advance the Claims watermark only after the Silver MERGE succeeds.
5. Verify both the audit record and persisted control state.

#### Expected Result

- Previous watermark: `2027-10-21`
- New watermark: `2027-10-23`
- Audit status: **SUCCESS**
- Source count: **2**
- Insert count: **1**
- Update count: **1**
- Reject count: **0**

The watermark is advanced only after successful Silver processing, preserving
restart safety.

In [22]:
# ============================================================
# STEP 12F - COMMIT CONTROLLED CLAIMS AUDIT + WATERMARK
# ============================================================

from datetime import datetime
from pyspark.sql import Row
import uuid

# ------------------------------------------------------------
# 1. Calculate successful batch watermark
# ------------------------------------------------------------

CONTROLLED_NEW_WATERMARK = (
    controlled_silver_ready_df
    .agg(F.max("last_updated").alias("max_watermark"))
    .first()["max_watermark"]
)

assert CONTROLLED_NEW_WATERMARK is not None, \
    "Controlled Claims watermark cannot be NULL."

assert CONTROLLED_NEW_WATERMARK > CONTROLLED_START_WATERMARK, \
    "Controlled Claims watermark did not advance."

print("Controlled Claims watermark calculated.")
print("-----------------------------------------")
print(f"Previous watermark : {CONTROLLED_START_WATERMARK}")
print(f"New watermark      : {CONTROLLED_NEW_WATERMARK}")


# ------------------------------------------------------------
# 2. Create a new batch ID for this controlled execution
# ------------------------------------------------------------

CONTROLLED_BATCH_ID = str(uuid.uuid4())
CONTROLLED_RUN_END_TS = datetime.now()


# ------------------------------------------------------------
# 3. Write SUCCESS audit
# ------------------------------------------------------------

controlled_audit_record = [
    Row(
        batch_id=CONTROLLED_BATCH_ID,
        pipeline_name=PIPELINE_NAME,
        table_name=SOURCE_NAME,
        start_time=RUN_START_TS,
        end_time=CONTROLLED_RUN_END_TS,
        source_count=controlled_incremental_count,
        insert_count=controlled_insert_count,
        update_count=controlled_update_count,
        reject_count=controlled_reject_count,
        status="SUCCESS",
        error_message=None
    )
]

controlled_audit_df = spark.createDataFrame(
    controlled_audit_record,
    schema=audit_schema
)

(
    controlled_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print()
print("Controlled Claims SUCCESS audit written.")


# ------------------------------------------------------------
# 4. Advance watermark AFTER successful MERGE
# ------------------------------------------------------------

spark.sql(f"""
    UPDATE {CONTROL_TABLE}
       SET last_watermark = TIMESTAMP('{CONTROLLED_NEW_WATERMARK}'),
           _updated_ts = current_timestamp()
     WHERE source_name = '{SOURCE_NAME}'
       AND is_active = true
""")

print("Claims watermark advanced.")


# ------------------------------------------------------------
# 5. Verify audit
# ------------------------------------------------------------

controlled_audit_check_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == CONTROLLED_BATCH_ID)
)

assert controlled_audit_check_df.count() == 1, \
    "Expected exactly one controlled Claims audit record."

controlled_audit_row = controlled_audit_check_df.first()

assert controlled_audit_row["status"] == "SUCCESS"
assert controlled_audit_row["source_count"] == 2
assert controlled_audit_row["insert_count"] == 1
assert controlled_audit_row["update_count"] == 1
assert controlled_audit_row["reject_count"] == 0


# ------------------------------------------------------------
# 6. Verify control table
# ------------------------------------------------------------

controlled_control_check_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
)

controlled_control_row = controlled_control_check_df.first()

assert (
    controlled_control_row["last_watermark"]
    == CONTROLLED_NEW_WATERMARK
), "Claims watermark persistence failed."


print()
print("==========================================")
print(" CLAIMS AUDIT + WATERMARK COMMIT PASSED")
print("==========================================")
print(f"Batch ID           : {CONTROLLED_BATCH_ID}")
print(f"Source records     : {controlled_incremental_count}")
print(f"INSERT records     : {controlled_insert_count}")
print(f"UPDATE records     : {controlled_update_count}")
print(f"Rejected records   : {controlled_reject_count}")
print(f"Final watermark    : {CONTROLLED_NEW_WATERMARK}")
print(f"Status             : SUCCESS")

display(controlled_audit_check_df)
display(controlled_control_check_df)

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 24, Finished, Available, Finished, False)

Controlled Claims watermark calculated.
-----------------------------------------
Previous watermark : 2027-10-21 00:00:00
New watermark      : 2027-10-23 00:00:00

Controlled Claims SUCCESS audit written.
Claims watermark advanced.

 CLAIMS AUDIT + WATERMARK COMMIT PASSED
Batch ID           : 966d74bb-ae04-4af5-ba55-6823e652b081
Source records     : 2
INSERT records     : 1
UPDATE records     : 1
Rejected records   : 0
Final watermark    : 2027-10-23 00:00:00
Status             : SUCCESS


SynapseWidget(Synapse.DataFrame, 33a83eaa-8fb0-46d6-9b18-ef3569300d72)

SynapseWidget(Synapse.DataFrame, 8acdaafc-ad89-4fd8-917e-4dea1550fa83)

### Step 13 — Final Claims Restart / Idempotency Verification

This final test verifies that the Claims incremental pipeline is restart-safe.

After the successful controlled batch:

- Bronze contains the original Claims plus the controlled test records.
- Silver contains the successfully merged business state.
- The persisted Claims watermark is `2027-10-23`.

The pipeline is now restarted without adding any new Bronze records.

#### Expected Result

- Bronze rows: **1202**
- Silver rows: **1201**
- Stored watermark: **2027-10-23**
- Incremental records after watermark: **0**
- No additional INSERT or UPDATE is required.
- Silver row count remains unchanged.

A successful result proves that already processed Claims will not be processed again
after a restart.

In [23]:
# ============================================================
# STEP 13 - FINAL CLAIMS RESTART / IDEMPOTENCY VERIFICATION
# ============================================================

# Reload persisted control state
restart_control_row = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
    .first()
)

RESTART_WATERMARK = restart_control_row["last_watermark"]

# Current Bronze and Silver state
restart_bronze_df = spark.table(CONFIG_SOURCE_TABLE)
restart_silver_df = spark.table(CONFIG_TARGET_TABLE)

restart_bronze_count = restart_bronze_df.count()
restart_silver_count = restart_silver_df.count()

# Simulate a fresh incremental extraction
restart_incremental_df = (
    restart_bronze_df
    .withColumn(
        "_restart_watermark_ts",
        F.to_timestamp(F.col(CONFIG_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_restart_watermark_ts") > F.lit(RESTART_WATERMARK)
    )
)

restart_incremental_count = restart_incremental_df.count()

print("Final Claims restart test completed.")
print("----------------------------------------")
print(f"Bronze Claims rows    : {restart_bronze_count}")
print(f"Silver Claims rows    : {restart_silver_count}")
print(f"Stored watermark      : {RESTART_WATERMARK}")
print(f"Incremental records   : {restart_incremental_count}")

# Assertions
assert restart_bronze_count == 1202, \
    f"Expected 1202 Bronze Claims, found {restart_bronze_count}."

assert restart_silver_count == 1201, \
    f"Expected 1201 Silver Claims, found {restart_silver_count}."

assert restart_incremental_count == 0, \
    f"Expected 0 restart records, found {restart_incremental_count}."

assert RESTART_WATERMARK == CONTROLLED_NEW_WATERMARK, \
    "Persisted Claims watermark does not match final processed watermark."

print()
print("============================================")
print(" CLAIMS RESTART / IDEMPOTENCY TEST PASSED")
print("============================================")
print("No previously processed Claims were selected.")
print("No duplicate INSERT occurred.")
print("No repeated UPDATE occurred.")
print("Silver state remained unchanged.")

StatementMeta(, 034404ce-c212-4154-8c52-84afd44d705d, 25, Finished, Available, Finished, False)

Final Claims restart test completed.
----------------------------------------
Bronze Claims rows    : 1202
Silver Claims rows    : 1201
Stored watermark      : 2027-10-23 00:00:00
Incremental records   : 0

 CLAIMS RESTART / IDEMPOTENCY TEST PASSED
No previously processed Claims were selected.
No duplicate INSERT occurred.
No repeated UPDATE occurred.
Silver state remained unchanged.
